# Colab — Entrenar SVM clásica desde `classical_features.npz`

Notebook autocontenido. **Sube este `.ipynb` a Colab** (`https://colab.research.google.com` → File → Upload notebook) y dale Run All.

Lo que hace:

1. Te pide que subas `classical_features.npz` (~150 MB, lo tienes generado en tu PC).
2. Entrena AdditiveChi2 + LinearSVC OvR con la calidad completa (`sample_steps=2`, `n_jobs=-1`) — Colab tiene memoria suficiente.
3. Te descarga `classical_svm_artifact.pkl`.

Después en tu PC corres `scripts/import_colab_svm.py` para envolver el artifact en el formato que espera el resto del proyecto.

## 1. Imports (Colab ya trae numpy + sklearn)

In [ ]:
import time
import pickle
import numpy as np
from sklearn.kernel_approximation import AdditiveChi2Sampler
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import LinearSVC
import sklearn
print(f"sklearn: {sklearn.__version__}")
print(f"numpy:   {np.__version__}")

## 2. Subir el `classical_features.npz`

Click en el botón "Choose Files" y selecciona el archivo que tienes en tu PC en `data/processed/classical_features.npz`.

In [ ]:
from google.colab import files
uploaded = files.upload()
fname = list(uploaded.keys())[0]
print(f"Uploaded: {fname}  ({len(uploaded[fname])/1024/1024:.1f} MB)")

data = np.load(fname)
X, y = data['X'], data['y']
print(f"X shape: {X.shape}, dtype={X.dtype}")
print(f"y shape: {y.shape}, classes={np.unique(y).tolist()}")
unique, counts = np.unique(y, return_counts=True)
for c, n in zip(unique, counts):
    print(f"  class {c}: {n}")

## 3. Entrenar SVM

`sample_steps=2` da output dim = D × 3 (mejor que `=1` en calidad). `n_jobs=-1` aprovecha los cores de Colab — aquí no hay riesgo de OOM como en Mac Air.

In [ ]:
X = np.clip(X.astype(np.float32, copy=False), 0.0, None)

t0 = time.time()
print("Step 1: AdditiveChi2Sampler.fit_transform...")
sampler = AdditiveChi2Sampler(sample_steps=2)
X_t = sampler.fit_transform(X)
print(f"  Done in {time.time()-t0:.1f}s. Shape: {X.shape} -> {X_t.shape}  "
      f"({X_t.nbytes/1024/1024:.0f} MB)")

t1 = time.time()
print("Step 2: OvR LinearSVC fit (n_jobs=-1)...")
svm = OneVsRestClassifier(
    LinearSVC(C=1.0, dual='auto', max_iter=5000, random_state=42),
    n_jobs=-1,
)
svm.fit(X_t, y)
print(f"  Done in {time.time()-t1:.1f}s")
print(f"  Classes seen: {svm.classes_.tolist()}")
print(f"Total elapsed: {time.time()-t0:.1f}s")

## 4. Guardar el artifact (portable)

Guardamos sampler + svm + lista de clases como un dict simple. Es portable entre máquinas (no depende del módulo `grocery_detection.classical.classifier`).

In [ ]:
target_class_ids = list(range(1, 21))  # 20 clases, ids 1..20 (orden de classes.yaml)
artifact = {
    'sampler': sampler,
    'svm': svm,
    'target_class_ids': target_class_ids,
    'sklearn_version': sklearn.__version__,
}
out_name = 'classical_svm_artifact.pkl'
with open(out_name, 'wb') as f:
    pickle.dump(artifact, f)
size_mb = __import__('os').path.getsize(out_name) / 1024 / 1024
print(f"Saved {out_name}  ({size_mb:.1f} MB)")

## 5. Descargar el artifact

In [ ]:
files.download(out_name)
print("Listo. Mete el .pkl en data/processed/ de tu PC y corre:")
print("  uv run python scripts/import_colab_svm.py")